# 06a · Atención: PatchTST e iTransformer (NeuralForecast)

| Modelo | Idea | Covariables |
|---|---|---|
| **PatchTST** | parte la ventana en *patches* (subsecuencias) que hacen de tokens de un Transformer; canal independiente (cada serie por separado, pesos compartidos); RevIN | no (solo y) |
| **iTransformer** | invierte el Transformer: cada **serie completa** es un token y la atención mezcla series entre sí (multivariado) | no (solo y) |

Mismo protocolo que los modelos propios (`src/models/nf.py`):
1. Se entrena con datos hasta 2014-09-30, usando julio–septiembre solo para early stopping, y se mide el MASE de validación con el backtest diario sobre julio–septiembre.
2. Se elige la mejor configuración.
3. Se reentrena una vez con todos los datos hasta el 30 de septiembre, durante los pasos que eligió la validación.
4. Se evalúa una sola vez en prueba.

La y se pasa normalizada por serie (mismo `SeriesScaler`) y se desnormaliza al final.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import matplotlib.pyplot as plt
from neuralforecast.models import PatchTST, iTransformer
from src import data as D, metrics as M, backtest as B
from src.models.nf import fit_nf, nf_forecast_fn

torch.set_float32_matmul_precision("high")
panel = D.build_panel()
scale = M.mase_scale(panel)
scaler = D.SeriesScaler().fit(panel)
z = scaler.transform(panel)[["unique_id", "ds", "y"]]
N_SERIES = z["unique_id"].nunique()
L, H = 168, 24

## 1. Configuraciones evaluadas en validación

In [ ]:
configs = {
    "PatchTST_p16": ("PatchTST", lambda **kw: PatchTST(h=H, input_size=L, patch_len=16, stride=8, encoder_layers=3,
                       n_heads=16, hidden_size=128, linear_hidden_size=256, dropout=0.2, revin=True,
                       learning_rate=5e-4, batch_size=32, windows_batch_size=1024, scaler_type="identity",
                       random_seed=42, **kw)),
    "PatchTST_p24": ("PatchTST", lambda **kw: PatchTST(h=H, input_size=L, patch_len=24, stride=12, encoder_layers=3,
                       n_heads=16, hidden_size=128, linear_hidden_size=256, dropout=0.2, revin=True,
                       learning_rate=5e-4, batch_size=32, windows_batch_size=1024, scaler_type="identity",
                       random_seed=42, **kw)),
    "iTransformer_128": ("iTransformer", lambda **kw: iTransformer(h=H, input_size=L, n_series=N_SERIES, hidden_size=128,
                       n_heads=8, e_layers=2, d_ff=256, dropout=0.1, learning_rate=5e-4, batch_size=32,
                       scaler_type="identity", random_seed=42, **kw)),
    "iTransformer_256": ("iTransformer", lambda **kw: iTransformer(h=H, input_size=L, n_series=N_SERIES, hidden_size=256,
                       n_heads=8, e_layers=3, d_ff=512, dropout=0.1, learning_rate=5e-4, batch_size=32,
                       scaler_type="identity", random_seed=42, **kw)),
}

val_rows, curvas = [], {}
for name, (col, make) in configs.items():
    nf, info = fit_nf(make, z, val_size=2208, max_steps=1500, val_check_steps=100, patience=5)
    fc_val = B.backtest(nf_forecast_fn(nf, scaler, L, col), panel, name=name,
                        start=D.VAL_START, end=D.VAL_END, verbose=False)
    r = M.evaluate(fc_val, panel, scale)["resumen"]
    curvas[name] = (info["train_traj"], info["valid_traj"])
    row = {k: v for k, v in info.items() if not k.endswith("_traj")}
    val_rows.append({"config": name, "modelo": col, **row, "val_MAE": r["MAE"], "val_MASE": r["MASE"]})
    print(f"{name:18s} MASE val {r['MASE']:.4f}  pasos {info['pasos']} (mejor {info['mejor_paso']})  "
          f"{info['fit_s']:.0f}s  params {info['params_total']:,}")
    del nf; torch.cuda.empty_cache()

val_tab = pd.DataFrame(val_rows).set_index("config")
val_tab

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, fam in zip(axes, ["PatchTST", "iTransformer"]):
    for name, (tr, va) in curvas.items():
        if not name.startswith(fam):
            continue
        tr_s = tr.set_index("step")["train_loss"].rolling(50, min_periods=1).mean()
        line, = ax.plot(tr_s.index, tr_s.values, ls="--", alpha=.6)
        ax.plot(va["step"], va["val_loss"], color=line.get_color(), marker="o", ms=3, label=name)
    ax.set_title(f"{fam}: pérdida MAE (-- train suavizada, — val)"); ax.set_xlabel("paso"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(D.ROOT / "outputs" / "figs" / "curvas_atencion.png", dpi=150)

## 2. Modelo final y evaluación en prueba

In [ ]:
filas = []
for fam in ["PatchTST", "iTransformer"]:
    cand = val_tab[val_tab["modelo"] == fam]
    best = cand["val_MASE"].idxmin()
    steps = int(cand.loc[best, "mejor_paso"])
    col, make = configs[best]
    print(f"{fam}: {best} (MASE val {cand.loc[best, 'val_MASE']:.4f}), reentrenando {steps} pasos con datos hasta 2014-09-30")
    nf, info = fit_nf(make, z, val_size=0, max_steps=steps)
    fc = B.backtest(nf_forecast_fn(nf, scaler, L, col), panel, name=fam)
    row, _ = B.finish(fam, fc, panel, scale, familia="Atención", L=L, estrategia="multi-salida",
                      covariables=False, config=best, params_total=info["params_total"],
                      params_entrenables=info["params_entrenables"],
                      train_s=float(cand.loc[best, "fit_s"]) + info["fit_s"],
                      val_MASE=float(cand.loc[best, "val_MASE"]),
                      gpu_mem_pico_MB=max(float(cand.loc[best, "gpu_mem_pico_MB"]), info["gpu_mem_pico_MB"]))
    filas.append(row)
    del nf; torch.cuda.empty_cache()

tabla = pd.DataFrame(filas).set_index("modelo")
tabla[["config", "params_total", "val_MASE", "MAE", "RMSE", "WAPE", "MASE", "MASE_navidad", "train_s", "inference_s"]]